# Phase 6: Data-Constrained Robustness & Comprehensive Benchmarks
### Evaluating Performance across 100%, 50%, and 20% Data Regimes & Final Reporting

This notebook evaluates model robustness under historical data constraints:
1. Loading cached graph structures and normalized sequences.
2. Defining training regimes: 100% (~80 days), 50% (~40 days), and 20% (~16 days).
3. Training and evaluating `TrafficGCN-Delta` across all three data availability regimes.
4. Training and evaluating classical baselines (LR & RF) on identical data subsets.
5. Synthesizing the final comparative benchmark table and exporting `final_results.csv`.
6. End-to-end automated pipeline and checkpointing utilities.


## Section 1: Environment Setup, Dependencies & System Configuration
We configure the runtime environment, set plotting styles, verify GPU hardware acceleration, and import fundamental scientific computing libraries (`numpy`, `pandas`, `matplotlib`).


In [4]:
#Day01_Traffic_Exploration.ipynb
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

print("Setup complete ")


Setup complete 


In [ ]:
# Ingest METR-LA dataset (checks local directory or Colab environment)
import os
import zipfile
import pandas as pd

file_path = 'METR-LA.h5'
zip_path = 'METR-LA.h5.zip'

if not os.path.exists(file_path):
    if os.path.exists(zip_path):
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
    elif os.path.exists('/content/METR-LA.h5.zip'):
        !unzip -o /content/METR-LA.h5.zip -d /content/
        file_path = '/content/METR-LA.h5'

if os.path.exists(file_path):
    df = pd.read_hdf(file_path)
elif os.path.exists('/content/METR-LA.h5'):
    df = pd.read_hdf('/content/METR-LA.h5')
else:
    # Fallback to local or downloaded copy
    df = pd.read_hdf('METR-LA.h5')

print("Dataset Shape (timesteps, sensors):", df.shape)
df.head()


In [ ]:
import urllib.request

base = "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/"
files = ["distances_la_2012.csv", "graph_sensor_ids.txt", "graph_sensor_locations.csv"]

for fname in files:
    urllib.request.urlretrieve(base + fname, fname)
    print(f"downloaded {fname}")


downloaded distances_la_2012.csv
downloaded graph_sensor_ids.txt
downloaded graph_sensor_locations.csv


In [ ]:
import numpy as np
import pandas as pd

with open('graph_sensor_ids.txt') as f:
    sensor_ids = f.read().strip().split(',')
num_sensors = len(sensor_ids)
id_to_idx = {sid: i for i, sid in enumerate(sensor_ids)}

dist_df = pd.read_csv('distances_la_2012.csv', dtype={'from': str, 'to': str})
print(dist_df.columns.tolist())  # sanity check: should be ['from', 'to', 'cost']

dist_mx = np.full((num_sensors, num_sensors), np.inf)

froms = dist_df['from'].values
tos = dist_df['to'].values
costs = dist_df['cost'].values

for f, t, c in zip(froms, tos, costs):
    if f in id_to_idx and t in id_to_idx:
        dist_mx[id_to_idx[f], id_to_idx[t]] = c

distances = dist_mx[~np.isinf(dist_mx)].flatten()
std = distances.std()
adj_mx = np.exp(-np.square(dist_mx / std))
adj_mx[adj_mx < 0.1] = 0
np.fill_diagonal(adj_mx, 1.0)

print(adj_mx.shape)
print((adj_mx > 0).sum(), "edges (incl. self-loops)")


['from', 'to', 'cost']
(207, 207)
1722 edges (incl. self-loops)


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

!pip install torch_geometric --no-deps -q

import torch

edge_index = np.array(np.nonzero(adj_mx))
edge_weight = adj_mx[edge_index[0], edge_index[1]]

edge_index = torch.tensor(edge_index, dtype=torch.long)
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

print("edge_index shape:", edge_index.shape)   # (2, 1722)
print("edge_weight shape:", edge_weight.shape) # (1722,)


edge_index shape: torch.Size([2, 1722])
edge_weight shape: torch.Size([1722])


In [ ]:
# Define graph sequence extraction: (N_samples, num_sensors, seq_length)
SEQ_LENGTH = 12

def create_graph_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length].T)   # (num_sensors, seq_length)
        y.append(data[i+seq_length])       # (num_sensors,)
    return np.array(X), np.array(y)

# Impute zeros and compute statistics exclusively on the training partition
df_clean = df.replace(0, np.nan).ffill().bfill()
values_clean = df_clean.values

train_len = int(len(values_clean) * 0.7)
mean = values_clean[:train_len].mean()
std = values_clean[:train_len].std()
values_norm = (values_clean - mean) / std

# Generate spatiotemporal graph sequences
X_graph, y_graph = create_graph_sequences(values_norm, SEQ_LENGTH)

# Chronological partition: 70% Train, 15% Validation, 15% Test
n = len(X_graph)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train_g, y_train_g = X_graph[:train_end], y_graph[:train_end]
X_val_g, y_val_g     = X_graph[train_end:val_end], y_graph[train_end:val_end]
X_test_g, y_test_g   = X_graph[val_end:], y_graph[val_end:]

print(f"Train normalization stats -> Mean: {mean:.4f}, Std: {std:.4f}")
print("Graph Tensor Shapes:")
print("  X_train_g:", X_train_g.shape, "| y_train_g:", y_train_g.shape)
print("  X_val_g:  ", X_val_g.shape,   "| y_val_g:  ", y_val_g.shape)
print("  X_test_g: ", X_test_g.shape,  "| y_test_g: ", y_test_g.shape)


mean: 58.435151943038306 std: 13.098095807372092
X_graph: (34260, 207, 12) y_graph: (34260, 207)
train/val/test: (23982, 207, 12) (5139, 207, 12) (5139, 207, 12)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
!pip install torch_geometric -q
from torch_geometric.nn import GCNConv

class TrafficGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=1):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_weight):
        # x: (num_nodes, in_channels)
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        return x.squeeze(-1)  # (num_nodes,)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.4 MB/s eta 0:00:00


In [ ]:
def make_batch_delta(X_np, y_np, indices, edge_index, edge_weight):
    data_list = []
    for i in indices:
        x = torch.tensor(X_np[i], dtype=torch.float)      # (207, 12)
        last_val = x[:, -1]                                 # (207,) most recent speed per node
        y_delta = torch.tensor(y_np[i], dtype=torch.float) - last_val  # predict the CHANGE
        data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_weight,
                               y=y_delta, last_val=last_val))
    return Batch.from_data_list(data_list)

def train_gcn_delta(X_train_g, y_train_g, X_val_g, y_val_g, edge_index, edge_weight,
                     epochs=30, batch_size=64, lr=0.001, hidden=64, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TrafficGCN(in_channels=X_train_g.shape[2], hidden_channels=hidden, out_channels=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    edge_index_d = edge_index.to(device)
    edge_weight_d = edge_weight.to(device)
    n_train = X_train_g.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0

        for i in range(0, n_train, batch_size):
            idx = perm[i:i+batch_size]
            batch = make_batch_delta(X_train_g, y_train_g, idx, edge_index, edge_weight).to(device)

            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.edge_attr)  # predicted delta
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        avg_train_loss = total_loss / n_train

        model.eval()
        with torch.no_grad():
            val_idx = np.random.choice(X_val_g.shape[0], size=min(500, X_val_g.shape[0]), replace=False)
            val_batch = make_batch_delta(X_val_g, y_val_g, val_idx, edge_index, edge_weight).to(device)
            val_out = model(val_batch.x, val_batch.edge_index, val_batch.edge_attr)
            val_loss = criterion(val_out, val_batch.y).item()

        print(f"Epoch {epoch+1}/{epochs} - train MAE(delta): {avg_train_loss:.4f} - val MAE(delta): {val_loss:.4f}")

    return model

model = train_gcn_delta(X_train_g, y_train_g, X_val_g, y_val_g, edge_index, edge_weight, epochs=30, lr=0.001)


Epoch 1/30 - train MAE(delta): 0.2000 - val MAE(delta): 0.1884
Epoch 2/30 - train MAE(delta): 0.1924 - val MAE(delta): 0.1927
Epoch 3/30 - train MAE(delta): 0.1920 - val MAE(delta): 0.1887
Epoch 4/30 - train MAE(delta): 0.1919 - val MAE(delta): 0.1959
Epoch 5/30 - train MAE(delta): 0.1918 - val MAE(delta): 0.1944
Epoch 6/30 - train MAE(delta): 0.1918 - val MAE(delta): 0.1942
Epoch 7/30 - train MAE(delta): 0.1918 - val MAE(delta): 0.1904
Epoch 8/30 - train MAE(delta): 0.1917 - val MAE(delta): 0.1908
Epoch 9/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1876
Epoch 10/30 - train MAE(delta): 0.1917 - val MAE(delta): 0.1891
Epoch 11/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1874
Epoch 12/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1864
Epoch 13/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1891
Epoch 14/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1888
Epoch 15/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1923
Epoch 16/30 - train MAE(delta): 0.1916 - val MAE(

In [ ]:
def evaluate_gcn_delta(model, X_np, y_np, edge_index, edge_weight, mean, std, device, batch_size=64):
    model.eval()
    preds_all, actuals_all = [], []
    with torch.no_grad():
        for i in range(0, X_np.shape[0], batch_size):
            idx = np.arange(i, min(i+batch_size, X_np.shape[0]))
            batch = make_batch_delta(X_np, y_np, idx, edge_index, edge_weight).to(device)
            delta_pred = model(batch.x, batch.edge_index, batch.edge_attr)
            pred = batch.last_val + delta_pred          # reconstruct actual predicted speed (normalized)
            actual = batch.last_val + batch.y            # reconstruct actual true speed (normalized)
            preds_all.append(pred.cpu().numpy())
            actuals_all.append(actual.cpu().numpy())
    preds = np.concatenate(preds_all) * std + mean
    actuals = np.concatenate(actuals_all) * std + mean
    mae = np.mean(np.abs(preds - actuals))
    rmse = np.sqrt(np.mean((preds - actuals)**2))
    return mae, rmse, preds


## Section 12: Data-Constrained Robustness Experiments (100%, 50%, 20% Data Regimes)
In practical deployments, historical sensor readings may only be available for a few weeks or months.
To assess model robustness under data scarcity, we evaluate performance across three training history availability regimes:
- **100% Data**: Full training set (~80 days)
- **50% Data**: Half training set (~40 days)
- **20% Data**: Limited training set (~16 days)

We extract the *most recent* $k\%$ of training data chronologically to accurately reflect real-world onboarding of new road corridors.

### 12.1 Training `TrafficGCN-Delta` across 100%, 50%, and 20% Data Regimes


In [ ]:
def get_data_subset(X_train_g, y_train_g, fraction):
    n = int(len(X_train_g) * fraction)
    # take the MOST RECENT `n` samples, not random — mimics a shorter real deployment history
    return X_train_g[-n:], y_train_g[-n:]

results = {}
predictions = None # Initialize predictions variable

for frac, label in [(1.0, '100%'), (0.5, '50%'), (0.2, '20%')]:
    print(f"\n=== Training GCN at {label} data ===")
    X_sub, y_sub = get_data_subset(X_train_g, y_train_g, frac)
    print("Training samples:", X_sub.shape[0])

    model_frac = train_gcn_delta(X_sub, y_sub, X_val_g, y_val_g, edge_index, edge_weight,
                                   epochs=30, lr=0.001)
    mae, rmse, current_predictions = evaluate_gcn_delta(model_frac, X_test_g, y_test_g, edge_index, edge_weight,
                                     mean, std, device)
    print(f"GCN @ {label} -> MAE: {mae:.3f}, RMSE: {rmse:.3f}")
    results[('GCN', label)] = (mae, rmse)
    # Store predictions from the last iteration for saving later
    if label == '20%': # Assuming you want to save predictions from the final model trained
        predictions = current_predictions



=== Training GCN at 100% data ===
Training samples: 23982
Epoch 1/30 - train MAE(delta): 0.1977 - val MAE(delta): 0.1841
Epoch 2/30 - train MAE(delta): 0.1922 - val MAE(delta): 0.1907
Epoch 3/30 - train MAE(delta): 0.1919 - val MAE(delta): 0.1917
Epoch 4/30 - train MAE(delta): 0.1919 - val MAE(delta): 0.1899
Epoch 5/30 - train MAE(delta): 0.1917 - val MAE(delta): 0.1874
Epoch 6/30 - train MAE(delta): 0.1916 - val MAE(delta): 0.1864
Epoch 7/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1871
Epoch 8/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1912
Epoch 9/30 - train MAE(delta): 0.1915 - val MAE(delta): 0.1939
Epoch 10/30 - train MAE(delta): 0.1914 - val MAE(delta): 0.1949
Epoch 11/30 - train MAE(delta): 0.1914 - val MAE(delta): 0.1894
Epoch 12/30 - train MAE(delta): 0.1914 - val MAE(delta): 0.1838
Epoch 13/30 - train MAE(delta): 0.1914 - val MAE(delta): 0.1859
Epoch 14/30 - train MAE(delta): 0.1913 - val MAE(delta): 0.1887
Epoch 15/30 - train MAE(delta): 0.1913 - val MAE(delta

### 12.2 Evaluating Classical Baselines (LR & RF) on Identical Data Subsets


In [ ]:
def get_flat_subset(X_train, y_train, fraction):
    n = int(len(X_train) * fraction)
    return X_train[-n:], y_train[-n:]

for frac, label in [(1.0, '100%'), (0.5, '50%'), (0.2, '20%')]:
    X_sub, y_sub = get_flat_subset(X_train, y_train, frac)
    print(f"\n=== {label} data, n={X_sub.shape[0]} ===")

    lr = LinearRegression()
    lr.fit(X_sub, y_sub)
    pred_lr = lr.predict(X_test)
    mae_lr = mean_absolute_error(y_test, pred_lr)
    rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
    print(f"LR -> MAE: {mae_lr:.3f}, RMSE: {rmse_lr:.3f}")
    results[('LR', label)] = (mae_lr, rmse_lr)

    rf = RandomForestRegressor(n_estimators=15, max_depth=8, n_jobs=-1, random_state=42)
    rf.fit(X_sub, y_sub)
    pred_rf = rf.predict(X_test)
    mae_rf = mean_absolute_error(y_test, pred_rf)
    rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
    print(f"RF -> MAE: {mae_rf:.3f}, RMSE: {rmse_rf:.3f}")
    results[('RF', label)] = (mae_rf, rmse_rf)


### 12.3 Checkpointing & Model Artifact Storage
Optionally save model state dictionaries, predictions, and JSON result summaries to local disk or Google Drive.


In [ ]:
# Save model checkpoint, JSON metrics, and prediction arrays
import os
import json

output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. Save trained PyTorch model state
if 'model_frac' in locals():
    torch.save(model_frac.state_dict(), os.path.join(output_dir, "gcn_model.pth"))
    print(f"Model checkpoint saved to {output_dir}/gcn_model.pth")

# 2. Save results dictionary
if 'results' in locals():
    results_serializable = {
        f"{k[0]}_{k[1]}": {
            "MAE": float(v[0]),
            "RMSE": float(v[1])
        }
        for k, v in results.items()
    }
    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(results_serializable, f, indent=4)
    print(f"Results saved to {output_dir}/results.json")

# 3. Save predictions array
if 'predictions' in locals() and predictions is not None:
    np.save(os.path.join(output_dir, "predictions.npy"), predictions)
    print(f"Predictions saved to {output_dir}/predictions.npy")


## Section 13: Comparative Benchmark Analysis, Summary Table & Export
We synthesize the performance of all evaluated models across the three data availability regimes into a structured comparison table and export the results for publication reporting.


In [5]:
import pandas as pd

rows = []
for model in ['LR', 'RF', 'GCN']:
    row = {'Model': model}
    for label in ['100%', '50%', '20%']:
        mae, rmse = results[(model, label)]
        row[f'MAE_{label}'] = round(mae, 3)
        row[f'RMSE_{label}'] = round(rmse, 3)
    rows.append(row)

results_df = pd.DataFrame(rows)
print(results_df)
results_df.to_csv('final_results.csv', index=False)


NameError: name 'results' is not defined

## Section 14: End-to-End Modular Training Pipeline & Automated Checkpointing
*(Decomposed & Modularized from the unified script in Cell 58)*

This section provides a clean, modular execution pipeline that manages paths, loads cached adjacency matrices, runs multi-regime training with checkpointing, and logs results directly to `outputs/results.csv`.

### 14.1 Directory Configuration & Artifact Paths


In [ ]:
import os
import csv
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('.').resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RESULTS_CSV = OUTPUTS_DIR / "results.csv"

for d in (RAW_DIR, CACHE_DIR, CHECKPOINT_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Project directories initialized:")
print("  Raw Data:    ", RAW_DIR)
print("  Checkpoints: ", CHECKPOINT_DIR)
print("  Outputs:     ", OUTPUTS_DIR)


### 14.2 Results Logging & Adjacency Cache Utilities


In [ ]:
def append_result_row(model_name, data_pct, mae, rmse):
    """Append one row to outputs/results.csv, writing the header if the file is new."""
    file_exists = RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["model", "data_pct", "mae", "rmse"])
        writer.writerow([model_name, data_pct, round(float(mae), 3), round(float(rmse), 3)])
    print(f"Logged result for {model_name} @ {data_pct} to {RESULTS_CSV}")

print("Logging utility ready.")


### 14.3 Automated Pipeline Execution Function
Run training and evaluation sequentially across data regimes, saving state checkpoints for each regime.


In [ ]:
def run_complete_pipeline():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running pipeline on device: {device}")

    # Save normalization and graph attributes for deployment
    torch.save(
        {
            "edge_index": edge_index,
            "edge_weight": edge_weight,
            "mean": mean,
            "std": std,
            "seq_length": SEQ_LENGTH,
        },
        CACHE_DIR / "graph_and_norm_stats.pt",
    )
    print("Graph and normalization cache saved.")

    # Train + evaluate at each data level
    for frac, label in [(1.0, "100%"), (0.5, "50%"), (0.2, "20%")]:
        ckpt_path = CHECKPOINT_DIR / f"gcn_{label.replace('%', 'pct')}.pt"

        if ckpt_path.exists():
            print(f"\n=== Checkpoint already exists for GCN @ {label}: {ckpt_path} ===")
            continue

        print(f"\n=== Training GCN at {label} data ===")
        X_sub, y_sub = get_data_subset(X_train_g, y_train_g, frac)
        print("Training samples:", X_sub.shape[0])

        model_frac = train_gcn_delta(
            X_sub, y_sub, X_val_g, y_val_g, edge_index, edge_weight,
            epochs=30, lr=0.001, device=device,
        )

        mae, rmse, _ = evaluate_gcn_delta(
            model_frac, X_test_g, y_test_g, edge_index, edge_weight, mean, std, device
        )
        print(f"GCN @ {label} -> MAE: {mae:.3f}, RMSE: {rmse:.3f}")

        # Persist checkpoint
        torch.save(model_frac.state_dict(), ckpt_path)
        append_result_row("GCN", label, mae, rmse)

    print(f"\nPipeline complete. Results in {RESULTS_CSV}, checkpoints in {CHECKPOINT_DIR}")

# Uncomment below to execute:
# run_complete_pipeline()
